In [0]:
catalog = "fmcg"
bronze_schema = "Bronze"
silver_schema = "Silver"
gold_schema = "Gold"
audit_schema = "audit"

In [0]:
# ============================================================
# CONFIG
# ============================================================

base_path = "s3://comptransportation/landing"

TABLE_CONFIG = {
    "customers": {
        "source": "s3://sportsbar-landing/landing/customers/*",
        "format": "csv",
        "load_type": "full",
    },
    "gross_price": {
        "source": "s3://sportsbar-landing/landing/gross_price/*",
        "format": "csv",
        "load_type": "full",
    },
    "orders": {
        "source": "s3://sportsbar-landing/landing/orders/*",
        "checkpoint": "s3://sportsbar-landing/checkpoint/orders",
        "format": "csv",
        "load_type": "incremental",
    },
    "products": {
        "source": "s3://sportsbar-landing/landing/products/*",
        "format": "csv",
        "load_type": "full",
    },
}


In [0]:
from pyspark.sql import Row
from datetime import datetime,timezone
import pyspark.sql.functions as F

def write_audit(layer, table_name, row_count, status, message=""):
    """Append one audit row to the central audit table."""
    row = Row(
            run_ts=datetime.now(timezone.utc).isoformat(),
            run_date = datetime.now(timezone.utc).strftime("%Y-%m-%d"),
            layer=layer,
            table_name=table_name,
            row_count=row_count,
            status=status,
            message=message,
)
    (
        spark.createDataFrame([row])
        .write.format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{audit_schema}.pipeline_audit")
    )


def split_good_bad(df, condition, fail_reason):
    """Split DataFrame into passing and failing rows."""
    good = df.filter(condition)
    bad = (
        df.filter(~condition)
        .withColumn("fail_reason", F.lit(fail_reason))
    )
    return good, bad

dim_table = ["customers", "products", "gross_price"]
def save_quarantine(bad_df, source,table_name):
    """Write bad rows to quarantine table."""
    if bad_df.isEmpty():
        return

    cnt = bad_df.count()
    if(table_name in dim_table):
        spark.sql(f"truncate table {catalog}.{audit_schema}.dq_quarantine_{table_name}")

    (
        bad_df
        .withColumn("source_table", F.lit(source))
        .withColumn("quarantined_at", F.current_timestamp())
        .write.format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{audit_schema}.dq_quarantine_{table_name}")
    )

    print(f"Quarantined {cnt} rows from {source}")